# Snowflake 101 — Part 3

## Who can see what, and what the text is telling you

Two topics that look unrelated and are not. Both are about data you hold but should not treat casually: personal information you must restrict, and free text you have been throwing away.

The data is 300 pieces of merchant feedback, each with a rating, a stated theme, the merchant's own words, and a contact email address.

By the end you will have:

1. Found personal data sitting in the clear
2. Masked it without changing the table or any query that reads it
3. Proved the mask works by becoming a different role
4. Made Snowflake read all 300 pieces of feedback and tell you what is in them
5. Checked whether it agreed with the themes a human assigned

Allow about 40 minutes.

## 0. Check your account is ready

Same eleven checks as the previous notebook. Thirty seconds now beats discovering a
missing table halfway through an exercise.

In [ ]:
%%sql -r preflight
-- Every row must say PASS. FAIL rows sort to the top, so if the first row says PASS you
-- are ready. If any row says FAIL, tell your facilitator the CHECK_NAME.
EXECUTE IMMEDIATE FROM @FISERV_SETUP.PUBLIC.WORKSHOP/branches/main/labs/101/generators/99_verify_101.sql;

In [ ]:
%%sql -r ctx
-- Pins your role, database, schema and warehouse for the rest of the notebook.
USE ROLE ACCOUNTADMIN;
USE DATABASE FISERV_101_DB;
USE SCHEMA GOVERNED;
USE WAREHOUSE FISERV_101_WH;

## 1. The problem

Look at the table. Everything here is synthetic, but read the last column as though it were not.

In [ ]:
%%sql -r feedback_sample
-- The feedback corpus. CONTACT_EMAIL is the column that should worry you.
SELECT FEEDBACK_ID, MERCHANT_ID, FEEDBACK_DATE, RATING,
       FEEDBACK_THEME, CONTACT_EMAIL,
       LEFT(FEEDBACK_TEXT, 90) AS FEEDBACK_EXTRACT
FROM FISERV_101_DB.GOVERNED.MERCHANT_FEEDBACK
ORDER BY FEEDBACK_DATE
LIMIT 8;

Anyone who can query this table can read every contact address in it. There is no way to grant access to the feedback without also granting access to the addresses, because access in SQL is granted per column at best and usually per table.

## 2. Masking policies

A **masking policy** is a rule attached to a column that changes what the column returns depending on who is asking. The table is not modified, the data is not copied, and the queries do not change. Someone selects `CONTACT_EMAIL` and gets either the real value or a masked one, decided at query time by their role.

The policy already exists in this account. It was created for you and deliberately **not attached to anything**, because attaching it is your job.

In [ ]:
%%sql -r show_policy
-- The policy exists. There is no INFORMATION_SCHEMA view for masking policies,
-- so SHOW is how you find them.
SHOW MASKING POLICIES LIKE 'EMAIL_MASK' IN SCHEMA FISERV_101_DB.GOVERNED;

In [ ]:
%%sql -r policy_body
-- Read what it actually does before you attach it. Never attach a policy you
-- have not read: a policy that returns NULL for everyone is a silent outage.
SELECT GET_DDL('POLICY', 'FISERV_101_DB.GOVERNED.EMAIL_MASK') AS POLICY_DEFINITION;

It returns the real value for `ACCOUNTADMIN` and `SYSADMIN`, and replaces everything before the `@` with `****` for everyone else. The domain survives, which is often what you want: enough to know it is a merchant address, not enough to contact anybody.

Now attach it.

In [ ]:
%%sql -r apply_policy
-- Attach the policy to the column. This is a metadata change; it does not
-- rewrite the table and it takes effect immediately for every query.
ALTER TABLE FISERV_101_DB.GOVERNED.MERCHANT_FEEDBACK
  MODIFY COLUMN CONTACT_EMAIL
  SET MASKING POLICY FISERV_101_DB.GOVERNED.EMAIL_MASK;

In [ ]:
%%sql -r still_admin
-- You are still ACCOUNTADMIN, so nothing looks different yet. That is correct.
SELECT FEEDBACK_ID, CONTACT_EMAIL
FROM FISERV_101_DB.GOVERNED.MERCHANT_FEEDBACK
ORDER BY FEEDBACK_ID
LIMIT 5;

## 3. Become someone else

`FISERV_101_ANALYST` is a role that has been granted `SELECT` on this database and nothing more. It is the shape of role a reporting analyst would actually hold.

Switch to it and run the identical query.

In [ ]:
%%sql -r as_analyst
-- Same query, different role. Nothing about the SQL changed.
USE ROLE FISERV_101_ANALYST;
SELECT FEEDBACK_ID, CONTACT_EMAIL
FROM FISERV_101_DB.GOVERNED.MERCHANT_FEEDBACK
ORDER BY FEEDBACK_ID
LIMIT 5;

In [ ]:
%%sql -r analyst_still_works
-- The analyst has lost nothing else. Aggregates, joins and filters all still
-- work, because only the one column is masked.
SELECT FEEDBACK_THEME, COUNT(*) AS FEEDBACK_COUNT, ROUND(AVG(RATING), 2) AS AVG_RATING
FROM FISERV_101_DB.GOVERNED.MERCHANT_FEEDBACK
GROUP BY FEEDBACK_THEME
ORDER BY FEEDBACK_COUNT DESC;

In [ ]:
%%sql -r back_to_admin
-- Switch back. Later cells need ACCOUNTADMIN, so do not skip this.
USE ROLE ACCOUNTADMIN;
SELECT CURRENT_ROLE() AS ROLE_NOW;

In [ ]:
%%sql -r policy_refs
-- Which columns is this policy attached to? Answering this for every policy is
-- most of what a governance review consists of.
-- POLICY_REFERENCES is a table function, not a view.
SELECT POLICY_NAME, REF_ENTITY_NAME, REF_COLUMN_NAME, POLICY_STATUS
FROM TABLE(FISERV_101_DB.INFORMATION_SCHEMA.POLICY_REFERENCES(
    POLICY_NAME => 'FISERV_101_DB.GOVERNED.EMAIL_MASK'));

**Worth knowing:** the policy is attached to a *column*, not to a query, a view or a dashboard. Anything that reads `CONTACT_EMAIL` inherits it, including views built on top of this table, exports, and anything an AI agent queries on someone's behalf. That is the property that makes it worth doing: you cannot accidentally route around it by writing a different query.

## 4. The text you have been ignoring

So far you have used `RATING` and `FEEDBACK_THEME`, both of which someone had to fill in. `FEEDBACK_TEXT` is what the merchant actually said, and until recently the only way to use it at scale was to build a classifier.

**AISQL** is a set of SQL functions that call a language model on each row. They are ordinary functions: you use them in a `SELECT`, they take a column, and they return a value you can group and filter on. Three of them here:

- `AI_SENTIMENT` returns how positive or negative the text is
- `AI_CLASSIFY` sorts the text into categories you supply
- `AI_COMPLETE` answers an arbitrary prompt

They are charged per call, so the cells below work on a **stratified sample of 42 rows** — six from each of the seven themes — not all 300. Get into that habit now rather than after a surprise.

The stratification is not incidental. `FEEDBACK_ID` runs in theme order, so taking the first 42 rows would have handed you a single theme and nothing to compare against.

In [ ]:
%%sql -r sentiment
-- Sentiment on a stratified sample. AI_SENTIMENT returns an object, so reach
-- into it with a path and cast, exactly as you did with telemetry JSON in Part 2.
-- The sample takes 6 rows PER THEME rather than the first 42 rows overall.
-- FEEDBACK_ID happens to be ordered by theme, so a plain LIMIT 42 would have
-- returned one theme and nothing to compare.
CREATE OR REPLACE TABLE FISERV_101_DB.GOVERNED.FEEDBACK_SCORED AS
WITH sampled AS (
    SELECT FEEDBACK_ID, RATING, FEEDBACK_THEME, FEEDBACK_TEXT
    FROM FISERV_101_DB.GOVERNED.MERCHANT_FEEDBACK
    QUALIFY ROW_NUMBER() OVER (PARTITION BY FEEDBACK_THEME
                               ORDER BY FEEDBACK_ID) <= 6
)
SELECT
    FEEDBACK_ID,
    RATING,
    FEEDBACK_THEME,
    FEEDBACK_TEXT,
    AI_SENTIMENT(FEEDBACK_TEXT):categories[0].sentiment::VARCHAR AS SENTIMENT
FROM sampled;

In [ ]:
%%sql -r sentiment_vs_rating
-- Does the model's reading of the words agree with the star rating?
-- If they disagree badly, one of the two is not measuring what you think.
SELECT SENTIMENT, COUNT(*) AS FEEDBACK_COUNT, ROUND(AVG(RATING), 2) AS AVG_RATING
FROM FISERV_101_DB.GOVERNED.FEEDBACK_SCORED
GROUP BY SENTIMENT
ORDER BY AVG_RATING;

In [ ]:
# Average star rating per model-assigned sentiment. Bars should climb left to right.
import matplotlib.pyplot as plt

# SQL cell results are pandas in Legacy Notebooks but Snowpark DataFrames in Notebooks
# in Workspaces on Runtime 2.6 and above. Convert only when the method is there, so this
# cell works on either without assuming which runtime the attendee picked.
_as_pandas = getattr(sentiment_vs_rating, "to_pandas", None)
df = _as_pandas() if _as_pandas else sentiment_vs_rating.copy()
colour_for = {"negative": "#D45B90", "mixed": "#FF9F36",
              "neutral": "#8A999E", "positive": "#29B5E8"}
colours = [colour_for.get(str(s).lower(), "#11567F") for s in df["SENTIMENT"]]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(df["SENTIMENT"], df["AVG_RATING"], color=colours)
ax.set_ylabel("Average star rating")
ax.set_ylim(0, 5)
ax.set_title("Model sentiment against the rating the merchant gave")
for i, r in enumerate(df.itertuples()):
    ax.text(i, r.AVG_RATING, f"{r.AVG_RATING}  (n={r.FEEDBACK_COUNT})",
            ha="center", va="bottom")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### Checking the human labels

`FEEDBACK_THEME` was assigned when the feedback was logged. Ask the model to categorise the same text independently, then compare. Where they disagree, either the model is wrong or the original label was, and both are worth knowing.

In [ ]:
%%sql -r classify
-- Classify the text into the SAME theme vocabulary the table already uses,
-- then compare with the theme that was recorded at the time. Using the exact
-- same labels matters: a different vocabulary would make every row look like a
-- mismatch and tell you nothing.
SELECT
    FEEDBACK_THEME AS RECORDED_THEME,
    AI_CLASSIFY(FEEDBACK_TEXT,
                ['settlement_delay', 'fees_transparency', 'terminal_reliability',
                 'support_responsiveness', 'onboarding', 'reporting',
                 'general_positive']):labels[0]::VARCHAR AS MODEL_THEME,
    COUNT(*) AS FEEDBACK_COUNT
FROM FISERV_101_DB.GOVERNED.FEEDBACK_SCORED
GROUP BY RECORDED_THEME, MODEL_THEME
ORDER BY FEEDBACK_COUNT DESC;

Read that as a confusion matrix. Rows where `RECORDED_THEME` and `MODEL_THEME` match are agreement, and you should get agreement on roughly three quarters of them.

Look at the mismatches, and in particular at what happened to `general_positive`. Several of those were classified as `terminal_reliability` instead. Both labels are defensible: the merchant was being positive *about their terminals*. The recorded vocabulary mixes two different axes — six topics and one sentiment — so a comment that is positive about a topic can only be filed under one of them.

That is not a model error. It is a schema problem the model exposed, and it would have stayed invisible if you had only ever counted the recorded themes.

### One more: ask a question in English

In [ ]:
%%sql -r summarise
-- AI_COMPLETE over concatenated feedback. LISTAGG builds one prompt from many
-- rows, so this is a single model call rather than 40.
SELECT AI_COMPLETE('claude-sonnet-4-5',
    'You are reviewing merchant feedback for a payments acquirer. '
    || 'Below are verbatim comments. In no more than four sentences, British English, '
    || 'state the single most costly operational problem these merchants describe '
    || 'and the evidence for it. Do not list every complaint.\n\n'
    || LISTAGG(FEEDBACK_TEXT, '\n---\n')
) AS ANALYSIS
FROM FISERV_101_DB.GOVERNED.FEEDBACK_SCORED
WHERE SENTIMENT = 'negative';

## Hand-off — Cortex Playground

**Why:** the cell above committed to a model and a prompt before you knew either was any good. **Cortex Playground** lets you try prompts and compare models side by side against your own data, then take the version that worked back into SQL. It is a UI feature.

**Steps**

1. In Snowsight, go to **AI & ML → Studio**, then open **Cortex Playground**.
2. Pick `claude-sonnet-4-5` in the left model slot and a different model in the right slot.
3. Paste this prompt into both:

   > You are reviewing merchant feedback for a payments acquirer. Identify the single most costly operational problem and the evidence for it, in no more than four sentences, British English.

4. Attach some of your own data as context, choosing `FISERV_101_DB.GOVERNED.FEEDBACK_SCORED`.
5. Run both and compare the answers.
6. Now change one word in the prompt: replace **costly** with **frequent**, and run again.

**What you should see:** the two models produce different emphases from identical input, and the costly-versus-frequent change moves the answer more than swapping the model did. The most frequent complaint and the most expensive one are not the same complaint, and the prompt is what decides which you get told about.

**Docs:** https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-playground

**Return here** when you have compared both models and both prompts.

## What you did

- Found personal data in the clear and read a **masking policy** before attaching it
- Attached the policy to a column, then proved it by switching to `FISERV_101_ANALYST` and running the same SQL
- Confirmed the analyst lost only that one column, not the ability to work
- Listed what the policy is attached to with `POLICY_REFERENCES`
- Used **AI_SENTIMENT**, **AI_CLASSIFY** and **AI_COMPLETE** as ordinary SQL functions on a controlled sample
- Compared the model's themes against the recorded ones, and treated disagreement as a question rather than an error
- Compared two models and two prompts in **Cortex Playground**

## Where this goes

Tomorrow you build the whole thing end to end: a governed pipeline over 30 million rows, a semantic model so questions can be asked in English, and an agent that answers them and gets marked on whether it was right.

The row access policy you will meet in session 5 is the same idea as today's masking policy, applied to whole rows instead of one column, and it is the reason the agent gives two different people two different correct answers.